In [0]:
# ============================================================
# Project  : ABC E-Commerce
# Layer    : Silver
# Notebook : 10_Silver_Shipments
# Author   : Suriya
# Purpose  : Clean and Validate Shipment Data
# ============================================================

from pyspark.sql.functions import *

# ------------------------------------------------------------
# Step 1 : Read Bronze Shipment Table
# ------------------------------------------------------------

shipments_df = spark.table("workspace.default.bronze_shipments")

# ------------------------------------------------------------
# Step 2 : Verify Data
# ------------------------------------------------------------

shipments_df.show(5)

shipments_df.printSchema()

print("Total Records :", shipments_df.count())

shipments_df.describe().show()

# ------------------------------------------------------------
# Step 3 : Null Validation
# ------------------------------------------------------------

null_df = shipments_df.filter(
    col("shipment_id").isNull() |
    col("order_id").isNull() |
    col("shipment_status").isNull()
)

print("Null Records :", null_df.count())

null_df.show()

# ------------------------------------------------------------
# Step 4 : Remove Null Records
# ------------------------------------------------------------

valid_df = shipments_df.filter(
    col("shipment_id").isNotNull() &
    col("order_id").isNotNull() &
    col("shipment_status").isNotNull()
)

# ------------------------------------------------------------
# Step 5 : Duplicate Validation
# ------------------------------------------------------------

duplicate_df = (
    valid_df
    .groupBy("shipment_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate Records :", duplicate_df.count())

duplicate_df.show()

# ------------------------------------------------------------
# Step 6 : Remove Duplicate Records
# ------------------------------------------------------------

silver_df = valid_df.dropDuplicates(["shipment_id"])

# ------------------------------------------------------------
# Step 7 : Business Rule Validation
# Shipment Status should be one of the below values
# ------------------------------------------------------------

invalid_status_df = silver_df.filter(
    ~col("shipment_status").isin("Delivered", "Pending", "Shipped")
)

print("Invalid Shipment Status :", invalid_status_df.count())

invalid_status_df.show()

# ------------------------------------------------------------
# Step 8 : Remove Invalid Shipment Status
# ------------------------------------------------------------

silver_df = silver_df.filter(
    col("shipment_status").isin("Delivered", "Pending", "Shipped")
)

# ------------------------------------------------------------
# Step 9 : Add Audit Columns
# ------------------------------------------------------------

silver_df = (
    silver_df
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("data_source", lit("shipments.csv"))
)

# ------------------------------------------------------------
# Step 10 : Verify Final Data
# ------------------------------------------------------------

silver_df.show(5)

silver_df.printSchema()

print("Silver Record Count :", silver_df.count())

# ------------------------------------------------------------
# Step 11 : Write Silver Table
# ------------------------------------------------------------

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_shipments")

# ------------------------------------------------------------
# Step 12 : Verify Silver Table
# ------------------------------------------------------------

spark.sql("""
SELECT *
FROM workspace.default.silver_shipments
LIMIT 10
""").show()